# 04 · Forecasts: copies predict the cued trials

**What this notebook does.** It builds one copy of each real person from each fitted model (rungs 0, 1 and 2), and asks every copy to do the cued trials its real person did. It does this under three forecasts:

| Forecast | What the copies are told about the cue |
| --- | --- |
| **A · No effect** | Nothing. They treat cued trials exactly like basic trials. |
| **B · Earlier studies** | The cue makes you more patient: log(k) drops by 0.175. Nothing else changes. Also run with a smaller (−0.07) and a larger (−0.28) drop. |
| **C · Learned from other people** | The cue's average effect, measured in 35 other participants, never in the person being predicted. |

**Where forecast B's numbers come from.** Peters (2020) re-analysed six earlier experiments that used the same kind of cue. The average shift in log(k) ranged from −0.07 to −0.28 across them. The table with each experiment's exact value could not be accessed, so the main forecast uses the middle of that range (−0.175), and the two ends are run as a smaller and a larger case. A drop of 0.175 in log(k) multiplies k by 0.84: the discount rate falls by about 16%.

**Why 6 groups in forecast C.** 42 people split evenly into 6 groups of 7. Each group's forecast then uses the other 35 people (83% of the sample) to measure the cue's effect, which keeps the estimate stable. Fewer, bigger groups (for example two halves of 21) could put most of the few strong responders on one side by chance. More groups (down to one person each) would mean 42 fits per model for little gain. This is a judgement call, not a number from the literature; 5 to 10 groups is the usual range for this kind of split.

**What this notebook does not do.** It never compares the forecasts with what real people did on the cued trials. That is notebook 05. Keeping the two apart means the forecasts are fixed before anyone sees how well they do.

**One exception, clearly marked:** Forecast C has to measure the cue's effect somewhere. It uses the cued trials of the *other* participants only. The person being predicted is never included.

Rung 3 is not built yet, so this notebook covers rungs 0 to 2.

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, "../src")
import dd, forecast

# QUICK = True: 8 people, 20 repeats, short fits. Checks the code runs. Its numbers mean nothing.
QUICK = False
N_PEOPLE = 8 if QUICK else None
TAG = "quick" if QUICK else "basic"            # which notebook 02 fits to read
N_REPS = 20 if QUICK else 200                  # how many times each copy does its trials
N_FOLDS = 2 if QUICK else 6                    # forecast C: groups of people predicted in turn
CHAINS, WARMUP, SAMPLES = (2, 200, 200) if QUICK else (4, 500, 500)   # forecast C's small fits

# Forecast B: shift in log(k) from earlier studies (Peters, 2020: range -0.07 to -0.28; main value = midpoint)
B_SHIFTS = {"B_main": -0.175, "B_low": -0.07, "B_high": -0.28}

RUNGS = ["rung0", "rung1", "rung2"]
FORECASTS = {"rung0": ["A", "C"], "rung1": ["A", "B_main", "B_low", "B_high", "C"], "rung2": ["A", "B_main", "B_low", "B_high", "C"]}
OUT = dd.ROOT / "data" / "derived" / "forecasts" / TAG
OUT.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(4)

## 1. The trials

Basic trials are what the models learned from. Cued trials are what we want to predict. Future and past cues are pooled as "cued".

`min_rt` is each person-day's fastest basic response. Rung 2 needs it because non-decision time is capped at 95% of it, exactly as in the fitted model.

In [ ]:
d = dd.reindex(dd.load_trials(), N_PEOPLE)
basic = d[d.condition == "basic"].reset_index(drop=True)
cued = d[d.condition != "basic"].reset_index(drop=True)
min_rt = basic.groupby("u").rt.min().sort_index().values
basic.to_csv(OUT / "basic_trials.csv", index=False)
cued.to_csv(OUT / "cued_trials.csv", index=False)
print(f"{d.p.nunique()} people · {len(basic):,} basic trials · {len(cued):,} cued trials "
      f"({(cued.condition == 'future').sum():,} future, {(cued.condition == 'past').sum():,} past)")

42 people · 8,004 basic trials · 7,991 cued trials (4,011 future, 3,980 past)


## 2. How a copy makes its choices

For each repeat, the notebook:
1. Picks one plausible set of traits for every person from notebook 02's estimates. The model is never perfectly sure of anyone's traits, so each repeat uses a different plausible version. The spread across repeats reflects that uncertainty.
2. Applies the forecast's cue shift, if any, to the cued trials.
3. Lets each copy do its own person's trials, one trial at a time.

Rung 2 copies produce a choice and a response time. Rungs 0 and 1 have no model of time, so their response times are drawn from that person-day's own basic-trial response times. This gives notebook 05 a "no model of time" comparison point.

In [ ]:
draws = {r: dd.load_draws(r, TAG)["theta"] for r in RUNGS}       # draws x person-days x traits
rt_draw = forecast.rt_sampler(basic, rng)
def run(rung, trials, shift_for_rep=None):
    t0 = time.time()
    ch, rt = forecast.simulate_copies(rung, trials, draws[rung], min_rt, N_REPS, rng, shift_for_rep, rt_draw)
    return ch, rt, time.time() - t0
def save(rung, name, ch, rt):
    np.savez_compressed(OUT / f"{rung}_{name}.npz", choice=ch, rt=rt)

## 3. Copies redo the basic trials

Not a forecast. Notebook 05 needs each copy's basic-trial behaviour to work out each copy's predicted cue effect (cued minus basic), in the same way as for the real person.

In [ ]:
for rung in RUNGS:
    ch, rt, secs = run(rung, basic)
    save(rung, "basic", ch, rt)
    print(f"{rung}: basic trials simulated {N_REPS} times in {secs:.0f} s")

rung0: basic trials simulated 200 times in 0 s
rung1: basic trials simulated 200 times in 0 s
rung2: basic trials simulated 200 times in 55 s


## 4. Forecast A · No effect

The copies see the cued trials but are told nothing about the cue.

Forecast A can still predict *some* difference between basic and cued trials. The cued trials use a different set of delays from the basic ones, so the offers are not identical. Notebook 05 accounts for this.

In [ ]:
for rung in RUNGS:
    ch, rt, secs = run(rung, cued)
    save(rung, "A", ch, rt)
    print(f"{rung}: forecast A in {secs:.0f} s")

rung0: forecast A in 0 s
rung1: forecast A in 0 s
rung2: forecast A in 55 s


## 5. Forecast B · Earlier studies

On cued trials only, each copy's log(k) drops by a fixed amount. That makes it more patient. Every other trait stays at the person's basic-trial value, including caution.

**What this tests:** the cue enters as a shift in one named trait. The question for notebook 05 is whether everything else follows from that one change: choices at every reward size and delay, response times, and how much people differ.

Rung 0 has no patience trait, so Forecast B does not apply to it.

In [ ]:
for rung in ["rung1", "rung2"]:
    for name, shift in B_SHIFTS.items():
        ch, rt, secs = run(rung, cued, lambda r, s=shift: {"logk": s})
        save(rung, name, ch, rt)
        print(f"{rung}: forecast {name} (log k {shift:+.3f}) in {secs:.0f} s")

rung1: forecast B_main (log k -0.175) in 0 s
rung1: forecast B_low (log k -0.070) in 0 s
rung1: forecast B_high (log k -0.280) in 0 s
rung2: forecast B_main (log k -0.175) in 56 s
rung2: forecast B_low (log k -0.070) in 57 s
rung2: forecast B_high (log k -0.280) in 60 s


## 6. Forecast C · Learned from other people

People are split at random into groups (6 groups of 7 in the full run). For each group in turn:

1. **Measure** the cue's average effect using the *other* groups' cued trials. Each training person's basic-trial traits are held fixed at their estimates, and only the cue shift is estimated:
   - rung 0: a shift in the intercept;
   - rung 1: a shift in log(k);
   - rung 2: a shift in log(k) and in caution (log boundary).
2. **Predict** the left-out group: their copies do their cued trials with that shift applied. Each repeat uses a different plausible value of the shift, so the forecast carries its uncertainty.

The left-out group's cued trials are never used to measure the shift.

In [ ]:
folds = forecast.make_folds(d.person.unique(), N_FOLDS)
d["fold"] = d.person.map(folds); cued["fold"] = cued.person.map(folds)
print(pd.Series(folds).value_counts().sort_index().rename("people per group").to_frame().T)

                  1  2  3  4  5  6
people per group  7  7  7  7  7  7


In [ ]:
import cmdstanpy
theta_mean = {r: draws[r].mean(0) for r in RUNGS}       # each person-day's estimated traits
shift_rows, shift_draws = [], {}
LIMIT = {"intercept": 5, "logk": 2, "log_alpha": 1}     # the limits set in stan/cue_shift_*.stan
for rung in RUNGS:
    model = cmdstanpy.CmdStanModel(stan_file=str(dd.STAN_DIR / f"cue_shift_{rung}.stan"))
    for f in range(1, N_FOLDS + 1):
        train = cued[cued.fold != f]
        data = forecast.cue_shift_data(rung, train, theta_mean[rung], min_rt)
        n_capped = data.pop("_n_capped", 0)
        names = [f"d_{t}" for t in forecast.SHIFTED[rung]]
        start = {n: 0.0 for n in names}                 # every chain starts at "the cue changes nothing"
        fit, tries = None, 0
        while fit is None:                              # a rare chain failure gets a fresh random start (up to 3 tries)
            tries += 1
            try:
                fit = model.sample(data=data, chains=CHAINS, parallel_chains=CHAINS, iter_warmup=WARMUP, iter_sampling=SAMPLES,
                                   seed=100 * tries + f, inits=start, step_size=0.1, adapt_delta=0.95, show_progress=False)
            except RuntimeError:
                print(f"{rung}, group {f}: try {tries} failed" + ("; trying again with a different random start." if tries < 3 else "."))
                if tries == 3:
                    print("Stan's own messages follow:")
                    model.sample(data=data, chains=1, iter_warmup=20, iter_sampling=20, seed=f, inits=start, show_console=True)
                    raise
        sd = np.column_stack([fit.stan_variable(n) for n in names])
        shift_draws[(rung, f)] = sd
        s = fit.summary()
        for k, t in enumerate(forecast.SHIFTED[rung]):
            shift_rows.append({"rung": rung, "group predicted": f, "trait shifted": t, "training people": train.person.nunique(),
                               "shift": sd[:, k].mean(), "low (5%)": np.quantile(sd[:, k], .05), "high (95%)": np.quantile(sd[:, k], .95),
                               "R_hat": s.loc[f"d_{t}", "R_hat"], "tries": tries,
                               "near a limit": bool(np.quantile(np.abs(sd[:, k]), .99) > 0.9 * LIMIT[t]),
                               "person-days with lowered non-decision time": n_capped})
shifts = pd.DataFrame(shift_rows)
shifts.to_csv(OUT / "forecast_C_shifts.csv", index=False)
shifts.round(3)

23:45:42 - cmdstanpy - INFO - CmdStan start processing
23:45:42 - cmdstanpy - INFO - Chain [1] start processing
23:45:42 - cmdstanpy - INFO - Chain [2] start processing
23:45:42 - cmdstanpy - INFO - Chain [3] start processing
23:45:42 - cmdstanpy - INFO - Chain [4] start processing
23:45:44 - cmdstanpy - INFO - Chain [2] done processing
23:45:44 - cmdstanpy - INFO - Chain [3] done processing
23:45:44 - cmdstanpy - INFO - Chain [4] done processing
23:45:44 - cmdstanpy - INFO - Chain [1] done processing
23:45:44 - cmdstanpy - INFO - CmdStan start processing
23:45:44 - cmdstanpy - INFO - Chain [1] start processing
23:45:44 - cmdstanpy - INFO - Chain [2] start processing
23:45:44 - cmdstanpy - INFO - Chain [3] start processing
23:45:44 - cmdstanpy - INFO - Chain [4] start processing
23:45:47 - cmdstanpy - INFO - Chain [4] done processing
23:45:47 - cmdstanpy - INFO - Chain [3] done processing
23:45:47 - cmdstanpy - INFO - Chain [2] done processing
23:45:47 - cmdstanpy - INFO - Chain [1] do

,rung,group predicted,trait shifted,training people,shift,low (5%),high (95%),R_hat,tries,near a limit,person-days with lowered non-decision time
0,rung0,1,intercept,35,0.856,0.785,0.930,1.004,1,False,0
1,rung0,2,intercept,35,0.366,0.289,0.438,1.006,1,False,0
2,rung0,3,intercept,35,0.851,0.775,0.923,1.006,1,False,0
3,rung0,4,intercept,35,0.881,0.801,0.961,1.004,1,False,0
4,rung0,5,intercept,35,0.926,0.850,1.000,1.004,1,False,0
5,rung0,6,intercept,35,0.820,0.749,0.896,1.002,1,False,0
6,rung1,1,logk,35,-0.331,-0.380,-0.281,1.005,1,False,0
7,rung1,2,logk,35,-0.167,-0.211,-0.122,1.010,1,False,0
8,rung1,3,logk,35,-0.322,-0.368,-0.274,1.003,1,False,0
9,rung1,4,logk,35,-0.360,-0.412,-0.311,1.010,1,False,0


**How to read this table.** Each row is the cue's average effect as measured *without* one group, which is the number given to that group's copies.

- A negative shift in log(k) means the cue made people more patient.
- A positive shift in log boundary means the cue made people more careful.
- The rows for one rung should be similar across groups. If one group's row is very different, the effect depends a lot on who is in the training set (for example, whether the few strong responders were in it).
- "Tries": how many attempts the fit needed. Occasionally one of the four chains fails while the sampler is still tuning itself; the fit is then simply rerun from a different random start. Anything above 1 is noted, not a problem.
- "Near a limit": the shifts are limited to values far beyond any realistic cue effect (patience ±2 on the log scale, caution ±1, rung 0's intercept ±5), so the sampler cannot wander into impossible values. This column should say False everywhere. True would mean a limit is shaping the answer.
- "Person-days with lowered non-decision time": on some days a person's fastest cued response was faster than their non-decision time from the basic trials allows. A model cannot explain a response faster than its non-decision time, so for this small fit only, that person-day's non-decision time is lowered to 95% of their fastest cued response. This column counts how many person-days that affected in each training set.

In [ ]:
for rung in RUNGS:
    t0 = time.time()
    ch = np.zeros((N_REPS, len(cued)), dtype=np.int8); rt = np.zeros((N_REPS, len(cued)), dtype=np.float32)
    for f in range(1, N_FOLDS + 1):
        rows = np.flatnonzero(cued.fold.values == f)
        sd = shift_draws[(rung, f)]
        pick = rng.integers(0, len(sd), N_REPS)
        shift_for_rep = lambda r, sd=sd, pick=pick: dict(zip(forecast.SHIFTED[rung], sd[pick[r]]))
        c_f, r_f = forecast.simulate_copies(rung, cued.iloc[rows], draws[rung], min_rt, N_REPS, rng, shift_for_rep, rt_draw)
        ch[:, rows], rt[:, rows] = c_f, r_f
    save(rung, "C", ch, rt)
    print(f"{rung}: forecast C in {time.time() - t0:.0f} s")

rung0: forecast C in 1 s
rung1: forecast C in 1 s
rung2: forecast C in 62 s


## 7. What the copies predict (no real data in this table)

The average share of "later" choices the copies make on basic and on cued trials, under each forecast. **This is the prediction only.** It is compared with the real cued trials in notebook 05.

In [ ]:
rows = []
for rung in RUNGS:
    b = np.load(OUT / f"{rung}_basic.npz")
    for name in FORECASTS[rung]:
        f = np.load(OUT / f"{rung}_{name}.npz")
        bl, cl = b["choice"].mean(1), f["choice"].mean(1)               # per repeat
        row = {"rung": rung, "forecast": name, "basic: later": bl.mean(), "cued: later": cl.mean(),
               "predicted change (points)": 100 * (cl - bl).mean(),
               "range (95%)": f"{100*np.quantile(cl - bl, .025):+.1f} to {100*np.quantile(cl - bl, .975):+.1f}"}
        if rung == "rung2":
            row["predicted change in median RT (s)"] = float(np.mean(np.median(f["rt"], 1) - np.median(b["rt"], 1)))
        rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "forecast_summary.csv", index=False)
summary.round(3)

,rung,forecast,basic: later,cued: later,predicted change (points),range (95%),predicted change in median RT (s)
0,rung0,A,0.704,0.707,0.325,-0.9 to +1.6,NaN
1,rung0,C,0.704,0.764,6.060,+5.0 to +7.3,NaN
2,rung1,A,0.716,0.718,0.164,-1.0 to +1.4,NaN
3,rung1,B_main,0.716,0.734,1.828,+0.7 to +2.9,NaN
4,rung1,B_low,0.716,0.725,0.843,-0.4 to +1.9,NaN
5,rung1,B_high,0.716,0.744,2.754,+1.5 to +4.0,NaN
6,rung1,C,0.716,0.746,2.943,+1.8 to +4.0,NaN
7,rung2,A,0.720,0.723,0.293,-0.9 to +1.4,-0.003
8,rung2,B_main,0.720,0.740,2.045,+0.9 to +3.2,-0.007
9,rung2,B_low,0.720,0.730,0.988,-0.3 to +2.3,-0.005


## Saved files

Everything is in `data/derived/forecasts/basic/` (`quick/` for a quick run):

| File | Contents |
| --- | --- |
| `basic_trials.csv`, `cued_trials.csv` | The trials, in the order the simulations use |
| `rungX_basic.npz` | Copies redoing their basic trials |
| `rungX_A.npz`, `rungX_B_main.npz`, `rungX_B_low.npz`, `rungX_B_high.npz`, `rungX_C.npz` | Forecasts for the cued trials |
| `forecast_C_shifts.csv` | The cue effect measured in each training set |
| `forecast_summary.csv` | The table above |

Each `.npz` holds `choice` and `rt`: one row per repeat, one column per trial.